# NB06 — Final Evaluation & SHAP Explainability

**Project:** The Access Gap: Using Machine Learning to Map Who Cannot Get Care in Canada and Why
**Phase:** CRISP-DM Phase 5–6 — Evaluation + Deployment Prep
**Inputs:** NB05 tuned models, `X_test_pp_fe.npy` (**FIRST USE**), NB05 manifest
**Outputs:** Final test metrics, SHAP figures, Power BI export CSV

---

## ⚠️  TEST SET UNSEALED — Read Before Running

This notebook contains the **first and only use** of the held-out test set.
All modelling decisions (architecture, features, thresholds) were finalised in NB03–NB05
*before* this cell is executed. Running it constitutes the **final, unbiased performance estimate**.

> **Do not re-run NB05 after seeing test results** — that would constitute
> "peeking" at the test set, invalidating the unbiased estimate.

## Objectives

| Section | Content |
|---------|---------|
| 1 | Unseal test set — load & profile |
| 2 | Final evaluation of all tuned models on test set |
| 3 | SHAP global feature importance (summary + beeswarm) |
| 4 | SHAP dependence plots (top 3 features) |
| 5 | SHAP local explanations (waterfall for key individuals) |
| 6 | New Brunswick subgroup analysis |
| 7 | Power BI export — predictions + SHAP for dashboard |
| 8 | Final report metrics table |


In [ ]:
import subprocess, sys
REQUIRED = {
    'pyarrow':'pyarrow', 'pandas':'pandas', 'numpy':'numpy',
    'scikit-learn':'sklearn', 'xgboost':'xgboost', 'lightgbm':'lightgbm',
    'shap':'shap', 'matplotlib':'matplotlib', 'seaborn':'seaborn',
    'joblib':'joblib', 'scipy':'scipy',
}
for pkg, imp in REQUIRED.items():
    try: __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('Packages OK')


---
## 0. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import shap

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    confusion_matrix, classification_report,
)

warnings.filterwarnings('ignore')
shap.initjs()   # enable JS rendering for force plots in notebooks
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import config as cfg
from evaluation import evaluate_model, find_optimal_threshold, compare_models_table

plt.rcParams['figure.dpi']        = cfg.FIGURE_DPI
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
PROCESSED   = cfg.DATA_PROCESSED
MODELS_DIR  = cfg.MODELS_DIR
POWERBI_DIR = cfg.POWERBI_DIR
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

print(f'SHAP version : {shap.__version__}')
print(f'Root         : {PROJECT_ROOT}')


---
## 1. Unseal the Test Set

The test set has been untouched since it was split in NB02.
All modelling decisions are now finalised. This section loads it for the first time.


In [ ]:
# ── TEST SET — first load ─────────────────────────────────────────────────────
X_test_pp = np.load(PROCESSED / 'X_test_pp_fe.npy')
y_test    = pd.read_parquet(PROCESSED / 'y_test.parquet').squeeze().values

# ── Validation (for threshold reference — already seen in NB05) ───────────────
X_val_pp  = np.load(PROCESSED / 'X_val_pp_fe.npy')
y_val     = pd.read_parquet(PROCESSED / 'y_val.parquet').squeeze().values

# ── Training (for SHAP background) ───────────────────────────────────────────
X_train_pp = np.load(PROCESSED / 'X_train_pp_fe.npy')

# ── Feature names ─────────────────────────────────────────────────────────────
with open(PROCESSED / 'feature_names_pp_fe.json') as f:
    feature_names_raw = json.load(f)

# Clean names: strip ColumnTransformer prefix (e.g. 'ordinal__DHHGAGE' -> 'DHHGAGE')
feature_names = [n.split('__')[-1] for n in feature_names_raw]

# ── NB05 manifest ─────────────────────────────────────────────────────────────
try:
    with open(PROCESSED / 'nb05_manifest.json') as f:
        nb05_manifest = json.load(f)
    TUNED_MODELS_LIST = nb05_manifest['tuned_models']
    BEST_MODEL_NAME   = nb05_manifest['best_model_name']
    OPT_THRESHOLD     = nb05_manifest['best_opt_threshold']
    SAFE_NAMES        = nb05_manifest['model_safe_names']
    print('NB05 manifest loaded.')
except FileNotFoundError:
    # Fallback: NB04 models if NB05 was not run
    with open(PROCESSED / 'nb04_manifest.json') as f:
        nb04_manifest = json.load(f)
    TUNED_MODELS_LIST = nb04_manifest['top3_for_tuning']
    BEST_MODEL_NAME   = nb04_manifest['best_model']
    SAFE_NAMES        = nb04_manifest['model_safe_names']
    with open(PROCESSED / 'threshold_results_nb04.json') as f:
        thr04 = json.load(f)
    OPT_THRESHOLD = thr04[BEST_MODEL_NAME]['optimal_thr']
    print('WARNING: NB05 not found — using NB04 models.')

print(f'Test set     : {X_test_pp.shape}  at-risk: {y_test.mean()*100:.2f}%')
print(f'Val set      : {X_val_pp.shape}')
print(f'Train (bg)   : {X_train_pp.shape}')
print(f'Feature names: {len(feature_names)}')
print(f'Tuned models : {TUNED_MODELS_LIST}')
print(f'Best model   : {BEST_MODEL_NAME}')
print(f'Opt threshold: {OPT_THRESHOLD:.3f}')


In [ ]:
# ── Load all tuned models ─────────────────────────────────────────────────────
tuned_models = {}
for name in TUNED_MODELS_LIST:
    safe = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))
    # Try tuned first, fall back to base
    for suffix in ['_tuned', '']:
        path = MODELS_DIR / f'{safe}{suffix}.joblib'
        if path.exists():
            tuned_models[name] = joblib.load(path)
            print(f'  Loaded: {path.name}')
            break

# Also load best model reference
best_model = tuned_models[BEST_MODEL_NAME]

print(f'\nModels loaded: {list(tuned_models.keys())}')
print(f'Best model   : {BEST_MODEL_NAME}')


In [ ]:
# ── Test Set Profile ──────────────────────────────────────────────────────────
print('TEST SET PROFILE')
print('=' * 55)
print(f'  Rows             : {len(y_test):,}')
print(f'  Features         : {X_test_pp.shape[1]}')
print(f'  At-risk (class=1): {y_test.sum():,}  ({y_test.mean()*100:.2f}%)')
print(f'  Not-at-risk (0)  : {(1-y_test).sum():,}  ({(1-y_test).mean()*100:.2f}%)')
print(f'  Class ratio      : {(1-y_test).sum() / y_test.sum():.2f}:1')
print()
print('  This distribution should match NB02 split statistics.')
print('  Any large deviation suggests a data pipeline error.')


---
## 2. Final Test Set Evaluation

Each tuned model is evaluated on the test set using its **optimal threshold**
(found on the validation set in NB05). This is the final, reportable performance.

> These numbers go directly into the capstone report.
> Do not revise model choices after seeing them.


In [ ]:
# ── Evaluate each tuned model at its optimal threshold ────────────────────────

# Load per-model optimal thresholds
try:
    with open(PROCESSED / 'threshold_results_nb05.json') as f:
        thr_results = json.load(f)
except FileNotFoundError:
    with open(PROCESSED / 'threshold_results_nb04.json') as f:
        thr_results = json.load(f)

test_results = {}

for name, model in tuned_models.items():
    opt_thr = thr_results[name]['optimal_thr'] if name in thr_results else OPT_THRESHOLD
    safe    = SAFE_NAMES.get(name, name.lower().replace(' ', '_'))

    print(f'\n{"="*65}')
    print(f'  {name} — Final Test Evaluation  (threshold={opt_thr:.3f})')
    print(f'{"="*65}')

    result = evaluate_model(
        model, X_test_pp, y_test,
        model_name=f'{name} (Test)',
        threshold=opt_thr,
        save_path=cfg.FIGURES_DIR / f'nb06_test_eval_{safe}.png',
    )
    test_results[name] = result

print('\nFinal test evaluation complete.')


In [ ]:
COLORS = {
    'Logistic Regression': '#FF9800',
    'Random Forest':       '#4CAF50',
    'XGBoost':             '#2196F3',
    'LightGBM':            '#9C27B0',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, model in tuned_models.items():
    y_prob = model.predict_proba(X_test_pp)[:, 1]
    color  = COLORS.get(name, '#757575')

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5,
                 label=f'{name.replace("Logistic Regression","LR").replace("Random Forest","RF")} (AUC={auc:.3f})')

    # PR
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec_arr, prec_arr, color=color, linewidth=2.5,
                 label=f'{name.replace("Logistic Regression","LR").replace("Random Forest","RF")} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--', alpha=0.4, linewidth=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — TEST SET', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=9); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1.02])

axes[1].axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'No-skill baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves — TEST SET', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=9); axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1.05])

plt.suptitle('Final Model Performance — Held-Out Test Set',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb06_test_roc_pr.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()


In [ ]:
# ── Final test metrics table ─────────────────────────────────────────────────
rows = []
for name in tuned_models:
    r       = test_results[name]
    opt_thr = thr_results[name]['optimal_thr'] if name in thr_results else OPT_THRESHOLD
    rows.append({
        'Model':       name,
        'Threshold':   opt_thr,
        'F1':          r['f1'],
        'Precision':   r['precision'],
        'Recall':      r['recall'],
        'PR-AUC':      r['pr_auc'],
        'ROC-AUC':     r['roc_auc'],
        'TP':          r['tp'],
        'FP':          r['fp'],
        'TN':          r['tn'],
        'FN':          r['fn'],
    })

final_df = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False).reset_index(drop=True)
final_df.index += 1

print('FINAL TEST SET RESULTS — ALL TUNED MODELS')
print('=' * 95)
print(final_df.to_string())
print()
print(f'F1 >= 0.70 target: {"MET" if final_df["F1"].max() >= 0.70 else "NOT MET"}')
print(f'Best model (PR-AUC): {final_df.iloc[0]["Model"]}  PR-AUC={final_df.iloc[0]["PR-AUC"]:.4f}')


---
## 3. SHAP — Global Feature Importance

SHAP (SHapley Additive exPlanations) decomposes each prediction into a sum of feature contributions.
Unlike permutation importance, SHAP values satisfy theoretical axioms (efficiency, symmetry, dummy, additivity).

### Computation Notes
- **Explainer:** `shap.TreeExplainer` for tree models; `shap.LinearExplainer` for Logistic Regression
- **Sample:** SHAP is computed on the full test set for global plots; a subset for local plots
- **Output:** SHAP values for the positive (at-risk) class
- **Background:** 200 k-means cluster centres from training set (for TreeExplainer interventional mode)


In [ ]:
# ── SHAP computation for best model ──────────────────────────────────────────
model_type = BEST_MODEL_NAME   # e.g. 'LightGBM'
model      = best_model

print(f'Computing SHAP values for: {model_type}')
print(f'  Test set: {X_test_pp.shape}')

# Background data: 200 k-means summaries from training set (speeds up computation)
print('  Building background (200 k-means clusters from training)...')
background = shap.kmeans(X_train_pp, 200)

if model_type in ('Random Forest', 'XGBoost', 'LightGBM'):
    explainer   = shap.TreeExplainer(model, data=background,
                                     feature_perturbation='interventional')
    shap_values = explainer.shap_values(X_test_pp)
    # RF returns list [class0_shap, class1_shap]; XGB/LGB returns single array
    if isinstance(shap_values, list):
        sv = np.array(shap_values[1])          # positive (at-risk) class
        ev = explainer.expected_value[1]
    else:
        sv = np.array(shap_values)
        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)):
            ev = float(ev)
else:
    # Logistic Regression
    bg_sample = shap.sample(X_train_pp, 200, random_state=cfg.RANDOM_STATE)
    explainer  = shap.LinearExplainer(model, bg_sample)
    sv_raw     = explainer.shap_values(X_test_pp)
    sv = np.array(sv_raw[1]) if isinstance(sv_raw, list) else np.array(sv_raw)
    ev = (explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray))
          else float(explainer.expected_value))

print(f'  SHAP values shape : {sv.shape}')
print(f'  Expected value    : {ev:.4f}  (= global mean probability in log-odds space)')
print(f'  Mean |SHAP|       : {np.abs(sv).mean():.5f}')


In [ ]:
# ── SHAP Summary — Mean |SHAP| bar chart ─────────────────────────────────────
#
# Shows the AVERAGE magnitude of each feature's contribution across all test predictions.
# This is the most interpretable global importance measure for policy audiences.

fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(
    sv, X_test_pp,
    feature_names=feature_names,
    plot_type='bar',
    max_display=25,
    show=False,
)
plt.title(f'Global SHAP Feature Importance ({model_type})\n'
          f'Mean |SHAP value| across {len(y_test):,} test respondents',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb06_shap_bar.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()
print('SHAP bar chart saved.')


In [ ]:
# ── SHAP Beeswarm — Direction + Magnitude ────────────────────────────────────
#
# Each dot = one test respondent.
# X-axis = SHAP value (positive = increases predicted probability of being at-risk).
# Colour = feature value (red = high, blue = low).
# This reveals HOW each feature affects the prediction, not just how much.

fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(
    sv, X_test_pp,
    feature_names=feature_names,
    plot_type='dot',
    max_display=25,
    show=False,
)
plt.title(f'SHAP Beeswarm — Feature Impact Direction ({model_type})\n'
          f'Red = high feature value, Blue = low feature value',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb06_shap_beeswarm.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()
print('SHAP beeswarm saved.')


---
## 4. SHAP Dependence Plots — Top 3 Features

A dependence plot shows:
- **X-axis**: Raw feature value
- **Y-axis**: SHAP value for that feature (contribution to at-risk probability)
- **Colour**: The feature that most interacts with this one (auto-selected)

This answers: *"At what values of income / age / food-security does risk increase sharply?"*


In [ ]:
# ── Top 3 features by mean |SHAP| ────────────────────────────────────────────
mean_abs_shap = np.abs(sv).mean(axis=0)
top3_idx      = np.argsort(mean_abs_shap)[::-1][:3]
top3_names    = [feature_names[i] for i in top3_idx]

print(f'Top 3 features: {top3_names}')
print()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, feat_idx, feat_name in zip(axes, top3_idx, top3_names):
    plt.sca(ax)
    shap.dependence_plot(
        feat_idx, sv, X_test_pp,
        feature_names=feature_names,
        ax=ax,
        show=False,
    )
    ax.set_title(f'Dependence: {feat_name}', fontweight='bold', fontsize=11)

plt.suptitle('SHAP Dependence Plots — Top 3 Features\n'
             'Y-axis = contribution to at-risk probability; colour = strongest interactor',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb06_shap_dependence.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()
print('Dependence plots saved.')


---
## 5. SHAP Local Explanations — Individual Predictions

Waterfall plots show **why** the model assigned a specific probability to a specific individual.
The baseline (expected value) is adjusted up or down by each feature's SHAP contribution.

Three cases are examined:
1. **Highest-risk individual** — the respondent with the highest predicted at-risk probability
2. **Hardest false negative** — an at-risk individual the model missed (lowest probability among true positives)
3. **Typical low-risk individual** — median probability among true negatives


In [ ]:
y_prob_test = best_model.predict_proba(X_test_pp)[:, 1]

# Case indices
highest_risk_idx = int(np.argmax(y_prob_test))
true_pos_mask    = y_test == 1
if true_pos_mask.sum() > 0:
    probs_tp = y_prob_test.copy()
    probs_tp[~true_pos_mask] = 999
    hardest_fn_idx = int(np.argmin(probs_tp))
else:
    hardest_fn_idx = None

true_neg_mask = y_test == 0
if true_neg_mask.sum() > 0:
    probs_tn = y_prob_test.copy()
    probs_tn[~true_neg_mask] = 999
    typical_tn_idx = int(np.where(true_neg_mask)[0][len(np.where(true_neg_mask)[0]) // 2])
else:
    typical_tn_idx = None

cases = [
    (highest_risk_idx, 'Highest-Risk Individual', '#E53935'),
    (hardest_fn_idx,   'Hardest False Negative',  '#FF9800'),
    (typical_tn_idx,   'Typical Low-Risk',         '#42A5F5'),
]
cases = [(idx, label, color) for idx, label, color in cases if idx is not None]

n_cases = len(cases)
fig, axes = plt.subplots(1, n_cases, figsize=(8 * n_cases, 7))
if n_cases == 1:
    axes = [axes]

for ax, (idx, label, color) in zip(axes, cases):
    shap_row  = sv[idx]
    prob      = float(y_prob_test[idx])
    true_lab  = int(y_test[idx])

    # Sort by absolute SHAP descending, show top 12
    order     = np.argsort(np.abs(shap_row))[::-1][:12]
    feat_lab  = [feature_names[i] for i in order]
    feat_shap = shap_row[order]

    colors = ['#E53935' if v > 0 else '#2196F3' for v in feat_shap]
    ax.barh(feat_lab, feat_shap, color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{label}\nP(at-risk)={prob:.3f}  True label={true_lab}',
                 fontweight='bold', fontsize=10, color=color)
    ax.set_xlabel('SHAP value (contribution to at-risk probability)')

plt.suptitle(f'SHAP Waterfall — Individual Explanations ({model_type})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.FIGURES_DIR / 'nb06_shap_waterfall.png', bbox_inches='tight', dpi=cfg.FIGURE_DPI)
plt.show()

print('Waterfall plots saved.')
for idx, label, _ in cases:
    print(f'  {label}: idx={idx}  P={y_prob_test[idx]:.3f}  True={y_test[idx]}')


---
## 6. New Brunswick Deep-Dive

New Brunswick (GEOGPRV = 13) is the focal province for this project:
- Lowest physician-to-population ratio in Canada
- High proportion of rural and official-language-minority communities

This section evaluates model performance and SHAP patterns specifically for NB respondents.


In [ ]:
# ── Load raw test features to get province codes ─────────────────────────────
X_test_raw = pd.read_parquet(PROCESSED / 'X_test_raw.parquet')
# Ensure row alignment
assert len(X_test_raw) == len(y_test), 'Row count mismatch between X_test_raw and y_test'

NB_CODE = 13  # GEOGPRV code for New Brunswick
nb_mask = (X_test_raw['GEOGPRV'] == NB_CODE).values

print(f'New Brunswick test respondents: {nb_mask.sum():,} / {len(nb_mask):,}')
print(f'  NB at-risk rate    : {y_test[nb_mask].mean()*100:.2f}%')
print(f'  Canada at-risk rate: {y_test.mean()*100:.2f}%')

if nb_mask.sum() > 10:
    X_test_nb = X_test_pp[nb_mask]
    y_test_nb = y_test[nb_mask]
    opt_thr   = OPT_THRESHOLD

    y_prob_nb = best_model.predict_proba(X_test_nb)[:, 1]
    y_pred_nb = (y_prob_nb >= opt_thr).astype(int)

    f1_nb     = f1_score(y_test_nb, y_pred_nb, pos_label=1, zero_division=0)
    recall_nb = recall_score(y_test_nb, y_pred_nb, pos_label=1, zero_division=0)
    prec_nb   = precision_score(y_test_nb, y_pred_nb, pos_label=1, zero_division=0)
    ap_nb     = average_precision_score(y_test_nb, y_prob_nb)

    # Full Canada metrics for comparison
    y_prob_all = best_model.predict_proba(X_test_pp)[:, 1]
    y_pred_all = (y_prob_all >= opt_thr).astype(int)
    f1_all     = f1_score(y_test, y_pred_all, pos_label=1, zero_division=0)
    ap_all     = average_precision_score(y_test, y_prob_all)

    print(f'\n  NB PERFORMANCE vs CANADA (threshold={opt_thr:.3f})')
    print(f'  {"Metric":<15} {"New Brunswick":>15} {"All Canada":>12}')
    print('  ' + '-' * 44)
    print(f'  {"F1":<15} {f1_nb:>15.4f} {f1_all:>12.4f}')
    print(f'  {"Recall":<15} {recall_nb:>15.4f}')
    print(f'  {"Precision":<15} {prec_nb:>15.4f}')
    print(f'  {"PR-AUC":<15} {ap_nb:>15.4f} {ap_all:>12.4f}')
    print(f'  {"N respondents":<15} {nb_mask.sum():>15,} {len(y_test):>12,}')
else:
    print('Insufficient NB test samples for evaluation (< 10).')
    X_test_nb, y_test_nb = None, None


In [ ]:
# ── SHAP comparison: NB vs Rest of Canada ────────────────────────────────────
if nb_mask.sum() > 10:
    sv_nb    = sv[nb_mask]
    sv_roc   = sv[~nb_mask]   # Rest of Canada

    mean_abs_nb  = np.abs(sv_nb).mean(axis=0)
    mean_abs_roc = np.abs(sv_roc).mean(axis=0)

    # Top 15 features in NB
    top15 = np.argsort(mean_abs_nb)[::-1][:15]
    feat_labels = [feature_names[i] for i in top15]
    nb_vals  = mean_abs_nb[top15]
    roc_vals = mean_abs_roc[top15]

    fig, ax = plt.subplots(figsize=(12, 7))
    x  = np.arange(len(feat_labels))
    w  = 0.38
    ax.barh(x - w/2, nb_vals,  w, label='New Brunswick',   color='#E53935', edgecolor='white')
    ax.barh(x + w/2, roc_vals, w, label='Rest of Canada',  color='#42A5F5', edgecolor='white')
    ax.set_yticks(x); ax.set_yticklabels(feat_labels, fontsize=9)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title('Feature Importance: New Brunswick vs Rest of Canada\n'
                 'Differences reveal region-specific drivers of access barriers',
                 fontweight='bold', fontsize=12)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(cfg.FIGURES_DIR / 'nb06_shap_nb_vs_canada.png',
                bbox_inches='tight', dpi=cfg.FIGURE_DPI)
    plt.show()
    print('NB vs Canada SHAP comparison saved.')
else:
    print('Skipped NB SHAP comparison (insufficient sample).')


---
## 7. Power BI Export

A flat CSV is produced for Power BI ingestion. Each row is one test respondent.

**Columns included:**
- Raw feature values from `X_test_raw.parquet` (original CCHS codes)
- `province_name` — human-readable province
- `true_label` — actual at-risk status (0/1)
- `predicted_prob` — model's predicted probability
- `predicted_class` — binary classification at optimal threshold
- `shap_<feature>` for the **top 20 features** by mean |SHAP|

This enables Power BI to:
- Map at-risk rates and predictions by province/region
- Show feature contribution charts (SHAP waterfall by region)
- Filter by demographic group to identify disparity patterns


In [ ]:
# ── Build Power BI export dataframe ─────────────────────────────────────────
y_prob_export = best_model.predict_proba(X_test_pp)[:, 1]
y_pred_export = (y_prob_export >= OPT_THRESHOLD).astype(int)

# Base: raw feature values
pbi_df = X_test_raw.copy().reset_index(drop=True)

# Province name
pbi_df['province_name'] = pbi_df['GEOGPRV'].map(cfg.PROVINCE_MAP)

# Predictions
pbi_df['true_label']      = y_test
pbi_df['predicted_prob']  = np.round(y_prob_export, 4)
pbi_df['predicted_class'] = y_pred_export

# Top 20 SHAP features
top20_idx   = np.argsort(np.abs(sv).mean(axis=0))[::-1][:20]
top20_names = [feature_names[i] for i in top20_idx]

for i, feat_name in zip(top20_idx, top20_names):
    col = f'shap_{feat_name}'
    pbi_df[col] = np.round(sv[:, i], 5)

# Save
pbi_path = POWERBI_DIR / 'test_predictions_shap.csv'
pbi_df.to_csv(pbi_path, index=False)

print(f'Power BI export saved: {pbi_path}')
print(f'  Rows    : {len(pbi_df):,}')
print(f'  Columns : {len(pbi_df.columns)}')
print(f'  SHAP features included: {top20_names}')
print()

# Province-level summary for Power BI map
prov_summary = (
    pbi_df.groupby('province_name').agg(
        n_respondents      = ('true_label', 'count'),
        true_at_risk_pct   = ('true_label', lambda x: round(x.mean()*100, 2)),
        pred_at_risk_pct   = ('predicted_class', lambda x: round(x.mean()*100, 2)),
        mean_pred_prob     = ('predicted_prob', lambda x: round(x.mean(), 4)),
    )
    .reset_index()
    .sort_values('pred_at_risk_pct', ascending=False)
)
prov_path = POWERBI_DIR / 'province_summary.csv'
prov_summary.to_csv(prov_path, index=False)

print('Province summary saved:')
print(prov_summary.to_string(index=False))


---
## 8. Final Report Metrics

Clean summary table for the capstone report. Reports test-set performance
at the optimal threshold found on the validation set.


In [ ]:
# ── Print publication-ready metrics table ─────────────────────────────────────
print('FINAL TEST SET PERFORMANCE — THE ACCESS GAP PROJECT')
print('=' * 75)
print(f'Dataset : CCHS 2022 PUMF  (n={len(y_test):,} test respondents)')
print(f'Target  : healthcare_access_risk (binary, ~{y_test.mean()*100:.1f}% positive)')
print(f'Split   : 60/20/20 stratified  |  SMOTE + class_weight=balanced')
print()
print(f'  {"Model":<22}  {"Thr":>5}  {"F1":>7}  {"Rec":>7}  {"Prec":>7}  {"PR-AUC":>8}  {"ROC-AUC":>9}')
print('  ' + '-' * 73)

for name in tuned_models:
    r = test_results[name]
    opt_thr = thr_results[name]['optimal_thr'] if name in thr_results else OPT_THRESHOLD
    best_flag = ' *' if name == BEST_MODEL_NAME else ''
    print(f'  {name:<22}  {opt_thr:>5.3f}  {r["f1"]:>7.4f}  {r["recall"]:>7.4f}'
          f'  {r["precision"]:>7.4f}  {r["pr_auc"]:>8.4f}  {r["roc_auc"]:>9.4f}{best_flag}')

print()
print('  * = Best model (highest PR-AUC)')
print()
best_r = test_results[BEST_MODEL_NAME]
print(f'  Best model confusion matrix ({BEST_MODEL_NAME}):')
print(f'    TP={best_r["tp"]:,}  FP={best_r["fp"]:,}  TN={best_r["tn"]:,}  FN={best_r["fn"]:,}')
print(f'    Of {y_test.sum():,} at-risk individuals in the test set,')
print(f'    the model correctly identified {best_r["tp"]:,} ({best_r["tp"]/y_test.sum()*100:.1f}%).')
print()

f1_target_met = best_r['f1'] >= 0.70
print(f'F1 >= 0.70 target: {"MET " if f1_target_met else "NOT MET"}  (F1 = {best_r["f1"]:.4f})')


In [ ]:
# ── Save final test results ────────────────────────────────────────────────────
with open(PROCESSED / 'test_results_nb06.json', 'w') as f:
    json.dump(test_results, f, indent=2)

final_df.to_csv(PROCESSED / 'final_model_comparison_nb06.csv', index=False)

# Save SHAP values for reproducibility
np.save(PROCESSED / 'shap_values_test.npy', sv)
with open(PROCESSED / 'shap_config_nb06.json', 'w') as f:
    json.dump({
        'best_model':      BEST_MODEL_NAME,
        'expected_value':  float(ev),
        'feature_names':   feature_names,
        'n_test':          int(len(y_test)),
    }, f, indent=2)

print('Final results saved:')
for fname in ['test_results_nb06.json', 'final_model_comparison_nb06.csv',
              'shap_values_test.npy', 'shap_config_nb06.json']:
    p = PROCESSED / fname
    if p.exists():
        print(f'  {fname}: {p.stat().st_size/1024:.1f} KB')

print()
print('Figures (reports/figures/):')
for f in sorted(cfg.FIGURES_DIR.glob('nb06_*.png')):
    print(f'  {f.name}')

print()
print('Power BI (reports/powerbi/):')
for f in sorted(POWERBI_DIR.glob('*.csv')):
    print(f'  {f.name}: {f.stat().st_size/1024:.0f} KB')


---
## 9. Project Summary

### Pipeline Overview

| Notebook | Phase | Key Output |
|----------|-------|------------|
| NB01 | Data Understanding | Target definition, EDA, leakage audit |
| NB02 | Data Preparation | Stratified split, ColumnTransformer, SMOTE |
| NB03 | Feature Engineering | Composite indices, ODHF regional features, selection |
| NB04 | Model Development | 5 classifiers, CV, learning curves |
| NB05 | Hyperparameter Tuning | Optuna TPE, best params, tuned models |
| NB06 | Final Evaluation | Test-set metrics, SHAP, Power BI export |

### Anti-Leakage Checklist — Final Verification

- [x] Target defined before any modelling (NB01)
- [x] All DO* routing flags removed (NB01)
- [x] Pipeline fitted on training data only — never on val or test (NB02)
- [x] SMOTE applied after splitting — no minority leakage to val/test (NB02, NB03)
- [x] Target encoding of GEODGHR4 uses training labels only (NB03)
- [x] Feature selection (MI, RF importance) fitted on training only (NB03)
- [x] CV in NB04/NB05 on raw (non-SMOTE) training with `class_weight='balanced'` (NB04, NB05)
- [x] Test set loaded here for the first time (NB06)
- [x] Thresholds selected on validation set — applied to test set without re-optimising (NB05, NB06)

### Policy Recommendations

Based on the SHAP analysis and New Brunswick subgroup results:

1. **Socioeconomic vulnerability** is the strongest driver — income support programs
   and food security initiatives address the root cause, not just symptoms.
2. **Immigration status + language barrier** combination shows compounding risk —
   culturally adapted healthcare navigation services are warranted.
3. **Chronic condition burden** among younger adults (30–50) predicts access barriers
   more strongly than age alone — chronic disease management programs for working-age adults.
4. **New Brunswick** consistently shows higher at-risk rates despite similar model performance —
   a provincial-level intervention is warranted rather than relying on national averages.
